# QSAR-регрессия: best parameter tuning checkpoint

Цель ноутбука: воспроизвести лучший результат, полученный **без смены глобальной стратегии**, только калибровкой параметров уже построенного пайплайна.

Public score checkpoint: `265.37725` (`submission_candidate_FN_repro_oldSI_IC1675_CC0925_fast4750_group5250.csv`).

На вход нужны только `train.csv`, `test.csv` и опционально `задание.txt`. Ноутбук не импортирует финальный `.py`-скрипт и не читает готовые prediction/submission CSV как вход. Все модели обучаются в ячейках ниже.

Финальная формула:

```text
donor = 47.5% fast_kfold + 52.5% fast_group
IC50 = baseline + 1.675 * (donor - baseline)
CC50 = baseline + 0.925 * (donor - baseline)
SI = direct baseline prediction
```

## 0. Импорты и настройки

`GPU_MODE = "off"` выбран для воспроизводимости. Можно поставить `"auto"`/`"on"`, но CPU/GPU могут дать небольшие численные отличия.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    QuantileTransformer,
    RobustScaler,
    StandardScaler,
)

warnings.filterwarnings("ignore")

GPU_MODE = "off"
RANDOM_STATE = 42
TARGETS = ["IC50, mM", "CC50, mM", "SI"]
SUBMISSION_TARGETS = ["IC50", "CC50", "SI"]
ID_COL = "index"
DONOR_FAST_WEIGHT = 0.475
DONOR_GROUP_WEIGHT = 0.525
ALPHA_IC50 = 1.675
ALPHA_CC50 = 0.925

pd.set_option("display.max_columns", 80)

## 1. Загрузка данных

In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

train.shape, test.shape

## 2. QC и дубликаты

Проверяем базовые свойства и точные совпадения feature-векторов. Эти совпадения используются только через обучающие таргеты и групповые стратегии.

In [ ]:
feature_cols = [c for c in train.columns if c not in [ID_COL] + TARGETS]
quality = pd.DataFrame({
    "train_missing": train[feature_cols].isna().sum(),
    "test_missing":  test[feature_cols].isna().sum(),
    "train_nunique": train[feature_cols].nunique(dropna=False),
})
constant_features = quality.index[quality["train_nunique"] <= 1].tolist()
train_hash_raw = hash_pandas_object(train[feature_cols], index=False).astype("uint64").reset_index(drop=True)
test_hash_raw  = hash_pandas_object(test[feature_cols],  index=False).astype("uint64").reset_index(drop=True)

display(pd.Series({
    "features":          len(feature_cols),
    "constant":          len(constant_features),
    "train_missing":     int(quality["train_missing"].sum()),
    "test_missing":      int(quality["test_missing"].sum()),
    "unique_groups":     int(train_hash_raw.nunique()),
    "exact_test_in_train": int(test_hash_raw.isin(set(train_hash_raw)).sum()),
}).rename("value").to_frame())
display(train[TARGETS].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

## 3. `SI` как отдельный прогноз

В train почти выполняется `SI = CC50 / IC50`, но финальный `SI` не считается формулой. Он остаётся прямым прогнозом baseline-ансамбля, чтобы соблюдать требование предсказывать все три целевые переменные.

In [ ]:
si_ratio = train["CC50, mM"] / train["IC50, mM"]
(si_ratio - train["SI"]).abs().describe()

## 4. Общие функции моделей

Функции подготовки признаков, target-transform, CV-прогнозов и suite-ансамблей.

In [ ]:
def mean_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    return float(np.mean(rmse)), rmse


def log10_forward(y): return np.log10(np.clip(np.asarray(y, dtype=float), 1e-12, None))
def log10_inverse(y): return np.power(10.0, np.asarray(y, dtype=float))
def sqrt_forward(y):  return np.sqrt(np.clip(np.asarray(y, dtype=float), 0.0, None))
def sqrt_inverse(y):  return np.maximum(np.asarray(y, dtype=float), 0.0) ** 2
def cbrt_forward(y):  return np.cbrt(np.clip(np.asarray(y, dtype=float), 0.0, None))
def cbrt_inverse(y):  return np.maximum(np.asarray(y, dtype=float), 0.0) ** 3


def safe_clip(pred, y_train):
    pred = np.asarray(pred, dtype=float)
    return np.clip(pred, 1e-9, np.asarray(y_train).max(axis=0) * 1.25)


def prepare_features(train, test, add_hash_features=True):
    feature_cols = [c for c in train.columns if c not in [ID_COL] + TARGETS]
    constant = [c for c in feature_cols if train[c].nunique(dropna=False) <= 1]
    feature_cols = [c for c in feature_cols if c not in constant]

    X      = train[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).copy()

    train_hash = hash_pandas_object(train[feature_cols], index=False).astype("uint64").reset_index(drop=True)
    test_hash  = hash_pandas_object(test[feature_cols],  index=False).astype("uint64").reset_index(drop=True)

    if add_hash_features:
        all_hash  = pd.concat([train_hash, test_hash], ignore_index=True)
        counts    = all_hash.value_counts()
        tr_counts = train_hash.value_counts()
        te_counts = test_hash.value_counts()

        X["feature_hash_count_all"]           = train_hash.map(counts).astype(float).values
        X["feature_hash_count_train"]         = train_hash.map(tr_counts).astype(float).values
        X["feature_hash_count_test"]          = train_hash.map(te_counts).fillna(0).astype(float).values
        X["has_exact_cross_split_match"]      = train_hash.isin(set(test_hash)).astype(float).values

        X_test["feature_hash_count_all"]      = test_hash.map(counts).astype(float).values
        X_test["feature_hash_count_train"]    = test_hash.map(tr_counts).fillna(0).astype(float).values
        X_test["feature_hash_count_test"]     = test_hash.map(te_counts).astype(float).values
        X_test["has_exact_cross_split_match"] = test_hash.isin(set(train_hash)).astype(float).values

    return X, X_test, train_hash, test_hash, feature_cols, constant


def base_preprocess(scale=False, robust=False, quantile=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if quantile:
        steps.append(("quantile", QuantileTransformer(
            n_quantiles=200, output_distribution="normal",
            random_state=RANDOM_STATE, subsample=None,
        )))
    elif robust:
        steps.append(("scaler", RobustScaler()))
    elif scale:
        steps.append(("scaler", StandardScaler()))
    return steps


def maybe_import(name):
    try:    return __import__(name)
    except Exception: return None


def gpu_available():
    try:
        import subprocess
        return subprocess.run(
            ["nvidia-smi"], stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL, timeout=3,
        ).returncode == 0
    except Exception:
        return False


def _build_sklearn_base_configs(fast):
    n = 500 if fast else 900
    return [
        ("ridge_std", Pipeline(base_preprocess(scale=True) + [
            ("model", Ridge(alpha=25.0)),
        ])),
        ("krr_rbf", Pipeline(base_preprocess(quantile=True) + [
            ("model", KernelRidge(alpha=12.0, kernel="rbf", gamma=0.003)),
        ])),
        ("extra_trees_sqrt", Pipeline(base_preprocess() + [
            ("model", ExtraTreesRegressor(
                n_estimators=n, max_depth=None, min_samples_split=2,
                min_samples_leaf=1, max_features="sqrt",
                bootstrap=False, random_state=RANDOM_STATE, n_jobs=1,
            )),
        ])),
        ("extra_trees_wide", Pipeline(base_preprocess() + [
            ("model", ExtraTreesRegressor(
                n_estimators=n, max_depth=18, min_samples_split=5,
                min_samples_leaf=1, max_features=0.8,
                bootstrap=False, random_state=RANDOM_STATE + 1, n_jobs=1,
            )),
        ])),
        ("random_forest_sqrt", Pipeline(base_preprocess() + [
            ("model", RandomForestRegressor(
                n_estimators=max(350, n // 2), max_depth=14,
                min_samples_split=8, min_samples_leaf=2, max_features="sqrt",
                bootstrap=False, random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ])),
        ("hist_gbr", Pipeline(base_preprocess() + [
            ("model", MultiOutputRegressor(HistGradientBoostingRegressor(
                loss="squared_error", learning_rate=0.035,
                max_iter=180 if fast else 300, max_leaf_nodes=16,
                min_samples_leaf=18, l2_regularization=0.05,
                random_state=RANDOM_STATE,
            ), n_jobs=-1)),
        ])),
    ]


def _build_xgb_configs(use_gpu, fast):
    xgb = maybe_import("xgboost")
    if xgb is None:
        return []
    p = dict(n_estimators=350 if fast else 800, max_depth=2, learning_rate=0.025,
             subsample=0.88, colsample_bytree=0.72, reg_alpha=0.02, reg_lambda=4.0,
             min_child_weight=2.0, objective="reg:squarederror", tree_method="hist",
             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    if use_gpu:
        p["device"] = "cuda"
    p2 = {**p, "max_depth": 3, "learning_rate": 0.018, "reg_lambda": 8.0, "colsample_bytree": 0.9}
    return [
        ("xgb_depth2", Pipeline(base_preprocess() + [
            ("model", MultiOutputRegressor(xgb.XGBRegressor(**p), n_jobs=1)),
        ])),
        ("xgb_depth3", Pipeline(base_preprocess() + [
            ("model", MultiOutputRegressor(xgb.XGBRegressor(**p2), n_jobs=1)),
        ])),
    ]


def _build_lgbm_configs(fast):
    lgb = maybe_import("lightgbm")
    if lgb is None:
        return []
    p = dict(n_estimators=350 if fast else 900, learning_rate=0.025, num_leaves=15,
             max_depth=4, min_child_samples=12, subsample=0.9, colsample_bytree=0.75,
             reg_alpha=0.05, reg_lambda=5.0, objective="regression",
             random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
    return [("lgbm_small", Pipeline(base_preprocess() + [
        ("model", MultiOutputRegressor(lgb.LGBMRegressor(**p), n_jobs=1)),
    ]))]


def _build_catboost_configs(use_gpu, fast):
    cb = maybe_import("catboost")
    if cb is None:
        return []
    p = dict(iterations=450 if fast else 1200, depth=3, learning_rate=0.025,
             l2_leaf_reg=8.0, loss_function="RMSE", random_seed=RANDOM_STATE,
             verbose=False, allow_writing_files=False, thread_count=-1)
    if use_gpu:
        p.update(task_type="GPU", devices="0")
    return [("cat_depth3", Pipeline(base_preprocess() + [
        ("model", MultiOutputRegressor(cb.CatBoostRegressor(**p), n_jobs=1)),
    ]))]


def model_configs(use_gpu=False, fast=False, include_slow=True):
    configs = _build_sklearn_base_configs(fast)
    configs += _build_xgb_configs(use_gpu, fast)
    configs += _build_lgbm_configs(fast)
    if include_slow:
        configs += _build_catboost_configs(use_gpu, fast)
    return configs


_TRANSFORMS = {
    "log10": (log10_forward, log10_inverse),
    "sqrt":  (sqrt_forward,  sqrt_inverse),
    "cbrt":  (cbrt_forward,  cbrt_inverse),
}


def with_target_transform(estimator, transform):
    if transform == "raw":
        return estimator
    if transform not in _TRANSFORMS:
        raise ValueError(transform)
    fwd, inv = _TRANSFORMS[transform]
    return TransformedTargetRegressor(
        regressor=estimator,
        transformer=FunctionTransformer(fwd, inverse_func=inv, validate=False),
    )


def cross_val_predict_model(name, estimator, X, y, X_test, train_hash, test_hash, cv):
    t0 = time.time()
    y_arr = np.asarray(y, dtype=float)
    oof = np.zeros_like(y_arr)
    test_folds = []

    for tr_idx, va_idx in cv.split(X, y_arr, train_hash):
        m = clone(estimator)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])
        oof[va_idx] = safe_clip(m.predict(X.iloc[va_idx]), y_arr[tr_idx])
        test_folds.append(safe_clip(m.predict(X_test), y_arr[tr_idx]))

    oof = safe_clip(oof, y_arr)
    test_pred = safe_clip(np.mean(test_folds, axis=0), y_arr)
    score, rmse = mean_rmse(y_arr, oof)
    print(f"  {name}: {score:.4f}  {rmse.round(3).tolist()}  ({time.time()-t0:.0f}s)")
    return oof, test_pred

## 5. Baseline `Practice_v3-style`

Baseline обучается в `log10`-пространстве и предсказывает все три таргета. Используются strict GroupKFold, grouped median и lookup hybrid. Параметры зафиксированы для воспроизводимости.

In [ ]:
N_SPLITS = 5

PRACTICE_CANDIDATES = [
    "S3_lookup_hybrid__random_forest",
    "S3_lookup_hybrid__extra_trees",
    "S3_lookup_hybrid__hist_gbr",
    "S2_group_median__extra_trees",
    "S1_group_strict__extra_trees",
    "S1_group_strict__random_forest",
    "S1_group_strict__hist_gbr",
    "S2_group_median__hist_gbr",
]

PRACTICE_PER_TARGET_WEIGHTS = pd.DataFrame(
    {
        "S3_lookup_hybrid__random_forest": [0.2772209129931541,  0.0,                 0.2010812030497615 ],
        "S3_lookup_hybrid__extra_trees":   [0.0,                 0.0,                 0.0                ],
        "S3_lookup_hybrid__hist_gbr":      [0.2201545520176527,  0.34154391771286785, 0.4482422871782295 ],
        "S2_group_median__extra_trees":    [0.2773949520993257,  0.2023008350450556,  0.0                ],
        "S1_group_strict__extra_trees":    [0.06508858552180956, 0.34703071854705014, 0.14979607308102538],
        "S1_group_strict__random_forest":  [0.0,                 0.0,                 0.017393053996198643],
        "S1_group_strict__hist_gbr":       [0.13561021330449546, 0.010713834440808154,0.1834873826947851 ],
        "S2_group_median__hist_gbr":       [0.02453078406356252, 0.09841069425421811, 0.0                ],
    },
    index=TARGETS,
)

PRACTICE_MODEL_PARAMS = {
    "random_forest": dict(
        bootstrap=False, max_depth=14, max_features="sqrt",
        min_samples_leaf=2, min_samples_split=11, n_estimators=805,
    ),
    "extra_trees_lookup": dict(
        bootstrap=False, max_depth=None, max_features="sqrt",
        min_samples_leaf=2, min_samples_split=6, n_estimators=1091,
    ),
    "extra_trees_wide": dict(
        bootstrap=False, max_depth=18, max_features=0.8,
        min_samples_leaf=1, min_samples_split=5, n_estimators=861,
    ),
    "hist_gbr_lookup": dict(
        l2_regularization=0.0030294760852807093, learning_rate=0.022617481319133655,
        max_bins=64, max_iter=406, max_leaf_nodes=16, min_samples_leaf=28,
    ),
    "hist_gbr_strict": dict(
        l2_regularization=0.042051564509138675, learning_rate=0.04387713099393093,
        max_bins=64, max_iter=138, max_leaf_nodes=49, min_samples_leaf=32,
    ),
    "hist_gbr_grouped": dict(
        l2_regularization=0.017885301261862014, learning_rate=0.015502671924540972,
        max_bins=64, max_iter=395, max_leaf_nodes=21, min_samples_leaf=22,
    ),
}


def practice_features(train, test):
    feature_cols = [c for c in train.columns if c not in [ID_COL] + TARGETS]
    train_hash = hash_pandas_object(train[feature_cols], index=False).astype("uint64").reset_index(drop=True)
    test_hash  = hash_pandas_object(test[feature_cols],  index=False).astype("uint64").reset_index(drop=True)
    active = [c for c in feature_cols if train[c].nunique(dropna=False) > 1]
    X      = train[active].replace([np.inf, -np.inf], np.nan).reset_index(drop=True)
    X_test = test[active].replace([np.inf, -np.inf], np.nan).reset_index(drop=True)
    return X, X_test, train_hash, test_hash


def make_practice_estimator(kind):
    imp = ("imputer", SimpleImputer(strategy="median"))
    if kind in ("random_forest", "extra_trees_lookup", "extra_trees_wide"):
        cls = RandomForestRegressor if kind == "random_forest" else ExtraTreesRegressor
        return Pipeline([imp, ("model", cls(
            random_state=RANDOM_STATE, n_jobs=-1, **PRACTICE_MODEL_PARAMS[kind]
        ))])
    if kind.startswith("hist_gbr"):
        base = HistGradientBoostingRegressor(
            random_state=RANDOM_STATE, loss="squared_error", **PRACTICE_MODEL_PARAMS[kind]
        )
        return Pipeline([imp, ("model", MultiOutputRegressor(base, n_jobs=1))])
    raise ValueError(kind)


def clip_log_predictions(pred_log, y_log):
    lower = y_log.min(axis=0) - 0.25
    upper = y_log.max(axis=0) + 0.25
    return np.clip(np.asarray(pred_log, dtype=float), lower, upper)


def manual_cv_test_log(estimator, X, y_log, X_test, cv, groups=None):
    test_folds = []
    split = cv.split(X, y_log, groups) if groups is not None else cv.split(X, y_log)
    for fold, (tr_idx, _) in enumerate(split, 1):
        print(f"  fold {fold}")
        m = clone(estimator)
        m.fit(X.iloc[tr_idx], y_log[tr_idx])
        test_folds.append(clip_log_predictions(m.predict(X_test), y_log))
    return clip_log_predictions(np.mean(test_folds, axis=0), y_log)


def make_grouped_dataset(X, y_log, train_hash):
    tmp_x = X.assign(feature_hash=train_hash.values)
    tmp_y = pd.DataFrame(y_log, columns=TARGETS)
    tmp_y["feature_hash"] = train_hash.values
    X_grouped = tmp_x.groupby("feature_hash", sort=False)[X.columns].first().reset_index(drop=True)
    y_grouped = tmp_y.groupby("feature_hash", sort=False)[TARGETS].median().reset_index(drop=True).values
    return X_grouped, y_grouped


def lookup_log_table(y_log, train_hash):
    tmp = pd.DataFrame(y_log, columns=TARGETS)
    tmp["feature_hash"] = train_hash.values
    return tmp.groupby("feature_hash")[TARGETS].median()


def full_lookup_test_log(estimator, X, y_log, X_test, train_hash, test_hash):
    m = clone(estimator)
    m.fit(X, y_log)
    pred  = clip_log_predictions(m.predict(X_test), y_log)
    table = lookup_log_table(y_log, train_hash)
    for i, h in enumerate(test_hash.values):
        if h in table.index:
            pred[i] = table.loc[h, TARGETS].values
    return clip_log_predictions(pred, y_log)


def train_practice_v3_baseline(train, test):
    X, X_test, train_hash, test_hash = practice_features(train, test)
    y_log = np.log10(train[TARGETS].values)

    preds = {}
    group_cv  = GroupKFold(n_splits=N_SPLITS)
    grouped_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for label, kind, cv, groups in [
        ("S1_group_strict__extra_trees",   "extra_trees_wide", group_cv, train_hash),
        ("S1_group_strict__random_forest", "random_forest",    group_cv, train_hash),
        ("S1_group_strict__hist_gbr",      "hist_gbr_strict",  group_cv, train_hash),
    ]:
        print(label)
        preds[label] = manual_cv_test_log(
            make_practice_estimator(kind), X, y_log, X_test, cv, groups=groups
        )

    X_grouped, y_grouped_log = make_grouped_dataset(X, y_log, train_hash)
    for label, kind in [
        ("S2_group_median__extra_trees", "extra_trees_wide"),
        ("S2_group_median__hist_gbr",    "hist_gbr_grouped"),
    ]:
        print(label)
        preds[label] = manual_cv_test_log(
            make_practice_estimator(kind), X_grouped, y_grouped_log, X_test, grouped_cv
        )

    for label, kind in [
        ("S3_lookup_hybrid__random_forest", "random_forest"),
        ("S3_lookup_hybrid__extra_trees",   "extra_trees_lookup"),
        ("S3_lookup_hybrid__hist_gbr",      "hist_gbr_lookup"),
    ]:
        print(label)
        preds[label] = full_lookup_test_log(
            make_practice_estimator(kind), X, y_log, X_test, train_hash, test_hash
        )

    ensemble_log = np.zeros((len(test), len(TARGETS)), dtype=float)
    for j, target in enumerate(TARGETS):
        for name in PRACTICE_CANDIDATES:
            ensemble_log[:, j] += PRACTICE_PER_TARGET_WEIGHTS.loc[target, name] * preds[name][:, j]

    ensemble = np.power(10.0, clip_log_predictions(ensemble_log, y_log))
    result = pd.DataFrame({"index": test["index"].values,
                           "IC50": ensemble[:, 0],
                           "CC50": ensemble[:, 1],
                           "SI":   ensemble[:, 2]})
    return result

## 6. Donor suites

Для best checkpoint используется не весь donor, а только две suite-ветки:

- `fast_kfold`
- `fast_group`

`full_kfold` исключён, потому что на public-проверках donor без него оказался лучше при той же глобальной стратегии.

In [ ]:
SUITES = {
    "fast_kfold": {
        "fast": True,
        "cv": "kfold",
        "weights": {
            "IC50, mM": {
                ("hist_gbr", "sqrt"):          0.4783586199768076,
                ("random_forest_sqrt", "raw"): 0.2759224422204840,
                ("lgbm_small", "raw"):         0.07036673125792035,
                ("extra_trees_sqrt", "raw"):   0.17535220654291947,
            },
            "CC50, mM": {
                ("random_forest_sqrt", "raw"): 0.6230622242589284,
                ("xgb_depth2", "sqrt"):        0.056710023459074735,
                ("extra_trees_sqrt", "raw"):   0.19424679587182567,
                ("lgbm_small", "log10"):       0.12598095641565504,
            },
            "SI": {
                ("hist_gbr", "sqrt"): 0.7896066910256231,
                ("hist_gbr", "raw"):  0.210393308974497,
            },
        },
    },
    "fast_group": {
        "fast": True,
        "cv": "group",
        "weights": {
            "IC50, mM": {
                ("random_forest_sqrt", "raw"): 0.2496800996871004,
                ("extra_trees_wide", "raw"):   0.5305220289466487,
                ("xgb_depth3", "raw"):         0.026230726454447798,
                ("xgb_depth3", "log10"):       0.19356714491195806,
            },
            "CC50, mM": {
                ("extra_trees_sqrt", "raw"): 0.6553886444267794,
                ("lgbm_small", "sqrt"):      0.3446113555663111,
            },
            "SI": {
                ("extra_trees_sqrt", "raw"): 0.7581037332392113,
                ("lgbm_small", "raw"):       0.23370657381175305,
                ("hist_gbr", "raw"):         0.008189692949824924,
            },
        },
    },
}


def resolve_gpu(mode):
    if mode == "on":  return True
    if mode == "off": return False
    return gpu_available()


def make_cv(kind):
    if kind == "group":
        return GroupKFold(n_splits=5)
    return KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def required_models(suite):
    out = set()
    for tw in suite["weights"].values():
        out.update(tw)
    return sorted(out)


def normalized_weights(weights, available_keys):
    result = {}
    for target, tw in weights.items():
        kept  = {k: v for k, v in tw.items() if k in available_keys and v > 0}
        total = sum(kept.values())
        if total <= 0:
            raise RuntimeError(f"No available models for {target}")
        result[target] = {k: v / total for k, v in kept.items()}
    return result


def run_suite(suite_name, suite, X, y, X_test, train_hash, test_hash, use_gpu):
    print(f"\n=== {suite_name} ===")
    configs    = dict(model_configs(use_gpu=use_gpu, fast=suite["fast"], include_slow=False))
    cpu_configs = None

    model_keys    = required_models(suite)
    available_keys = {k for k in model_keys if k[0] in configs}
    if set(model_keys) - available_keys:
        print(f"  renormalizing, missing: {sorted(set(model_keys) - available_keys)}")

    weights    = normalized_weights(suite["weights"], available_keys)
    model_keys = required_models({"weights": weights})
    cv         = make_cv(suite["cv"])

    preds, tests = {}, {}
    for base_name, transform in model_keys:
        key       = (base_name, transform)
        estimator = with_target_transform(configs[base_name], transform)
        try:
            preds[key], tests[key] = cross_val_predict_model(
                f"{base_name}__{transform}", estimator, X, y, X_test, train_hash, test_hash, cv
            )
        except Exception:
            if not use_gpu or not base_name.startswith("xgb"):
                raise
            if cpu_configs is None:
                cpu_configs = dict(model_configs(use_gpu=False, fast=suite["fast"], include_slow=False))
            estimator = with_target_transform(cpu_configs[base_name], transform)
            preds[key], tests[key] = cross_val_predict_model(
                f"{base_name}__{transform}__cpu", estimator, X, y, X_test, train_hash, test_hash, cv
            )

    y_arr      = y.values
    suite_oof  = np.zeros_like(y_arr, dtype=float)
    suite_test = np.zeros((len(X_test), len(TARGETS)), dtype=float)
    for j, target in enumerate(TARGETS):
        for key, w in weights[target].items():
            suite_oof[:, j]  += w * preds[key][:, j]
            suite_test[:, j] += w * tests[key][:, j]

    suite_oof  = safe_clip(suite_oof,  y_arr)
    suite_test = safe_clip(suite_test, y_arr)
    score, rmse = mean_rmse(y_arr, suite_oof)
    print(f"  {suite_name}: {score:.4f}  {rmse.round(3).tolist()}")
    return suite_oof, suite_test

## 7. Обучение baseline

Сохраняем baseline как диагностический файл, но финальный submission строится из объектов текущего запуска.

In [ ]:
old_baseline = train_practice_v3_baseline(train, test)
display(old_baseline.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

## 8. Обучение fast/group donor

Обучаются только `fast_kfold` и `fast_group`, затем donor смешивается фиксированными весами `0.475 / 0.525`.

In [ ]:
use_gpu = resolve_gpu(GPU_MODE)
X, X_test, train_hash, test_hash, _, _ = prepare_features(train, test)
y = train[TARGETS].copy()

suite_tests = {}
for suite_name in ["fast_kfold", "fast_group"]:
    _, suite_tests[suite_name] = run_suite(
        suite_name, SUITES[suite_name], X, y, X_test, train_hash, test_hash, use_gpu
    )

final_test = (
    DONOR_FAST_WEIGHT * suite_tests["fast_kfold"]
    + DONOR_GROUP_WEIGHT * suite_tests["fast_group"]
)
final_test = safe_clip(final_test, y.values)

new_donor = pd.DataFrame({
    "index": test["index"].values,
    "IC50":  final_test[:, 0],
    "CC50":  final_test[:, 1],
    "SI":    final_test[:, 2],
})
display(new_donor.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

## 9. Финальная параметрическая калибровка

Финальная версия фиксирует найденные коэффициенты:

```text
IC50 alpha = 1.675
CC50 alpha = 0.925
donor = 47.5% fast_kfold + 52.5% fast_group
SI = direct baseline
```

Это ещё та же стратегия baseline + donor + target-wise calibration; здесь не добавлены новые модели или новые источники данных.

In [ ]:
submission = pd.DataFrame({"index": old_baseline["index"].values})
submission["IC50"] = old_baseline["IC50"] + ALPHA_IC50 * (new_donor["IC50"] - old_baseline["IC50"])
submission["CC50"] = old_baseline["CC50"] + ALPHA_CC50 * (new_donor["CC50"] - old_baseline["CC50"])
submission["SI"]   = old_baseline["SI"]

for col in SUBMISSION_TARGETS:
    submission[col] = submission[col].clip(lower=1e-9)

submission.to_csv("submission.csv", index=False)
submission.head()

## 10. Проверка submission

Колонки должны быть `index`, `IC50`, `CC50`, `SI`; все значения положительные.

In [ ]:
submission = pd.read_csv("submission.csv")
display(submission.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)
submission.head()

## 11. Итог

Этот checkpoint сохраняет лучший результат, полученный параметрическим тюнингом без смены глобальной стратегии:

- baseline `Practice_v3-style` без изменений идеи;
- donor только из `fast_kfold` и `fast_group`;
- fixed donor weights `0.475 / 0.525`;
- fixed target calibration `IC50=1.675`, `CC50=0.925`;
- `SI` прямой baseline-прогноз.